In [ ]:
# Se cargan productos adquiridos para encontrar clientes con patrones similares.

import numpy as np
import pandas as pd

purchases = pd.read_csv("../data/customer_products.csv")
customer_product = pd.crosstab(purchases["customer_id"], purchases["product"])
customer_product

In [ ]:
# ¿Qué clientes se parecen más a C10 según los productos que ya tiene?

def cosine_similarity(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    normalized = np.divide(matrix, norms, out=np.zeros_like(matrix, dtype=float), where=norms != 0)
    return normalized @ normalized.T

similarity = pd.DataFrame(cosine_similarity(customer_product.to_numpy()), index=customer_product.index, columns=customer_product.index)
similarity.loc["C10"].sort_values(ascending=False)

In [ ]:
# Se estiman preferencias de productos no adquiridos con el aporte ponderado de vecinos.

target_vector = customer_product.loc["C10"]
neighbor_weights = similarity.loc["C10"].drop("C10")
predicted_scores = customer_product.drop("C10").T.dot(neighbor_weights) / neighbor_weights.sum()
recommendations = pd.DataFrame({"product": predicted_scores.index, "predicted_preference": predicted_scores.values, "already_owned": target_vector.values.astype(bool)})
recommendations = recommendations.loc[~recommendations["already_owned"]].sort_values("predicted_preference", ascending=False)
recommendations

In [ ]:
# Se conservan la matriz, los vecinos y las recomendaciones para verificar el resultado.

from pathlib import Path

submission_dir = Path("../submission")
customer_product.reset_index().to_csv(submission_dir / "customer_product_matrix.csv", index=False)
similarity.to_csv(submission_dir / "customer_similarity.csv")
neighbor_weights.sort_values(ascending=False).rename("similarity").reset_index().rename(columns={"index": "neighbor_customer"}).to_csv(submission_dir / "nearest_neighbors.csv", index=False)
recommendations.to_csv(submission_dir / "recommendations.csv", index=False)